In [1]:
import numpy as np
import csv


def trimf(x, a, b, c):
    x = np.asarray(x)
    y = np.zeros_like(x, dtype=float)
    idx = (a < x) & (x < b)
    y[idx] = (x[idx] - a) / (b - a)
    y[x == b] = 1.0
    idx = (b < x) & (x < c)
    y[idx] = (c - x[idx]) / (c - b)
    return np.clip(y, 0, 1)

def trapmf(x, a, b, c, d):
    x = np.asarray(x)
    y = np.zeros_like(x, dtype=float)
    idx = (a < x) & (x < b)
    y[idx] = (x[idx] - a) / (b - a)
    idx = (b <= x) & (x <= c)
    y[idx] = 1.0
    idx = (c < x) & (x < d)
    y[idx] = (d - x[idx]) / (d - c)
    return np.clip(y, 0, 1)

def classical_obesity_diagnosis(patient):
    """
    Klasyczna reguła: IF BMI >= 30 THEN Obesity
    """
    bmi = patient["bmi"]
    if bmi >= 30.0:
        return "Obesity (Otyłość)"
    else:
        return "Not Obese (Brak otyłości)"



# Definicje zbiorów rozmytych dla wejść (BMI)
def bmi_normalne(x):    return trapmf([x], 0, 0, 24, 26)[0]
def bmi_podwyzszone(x): return trapmf([x], 24, 26, 29, 31)[0]
def bmi_wysokie(x):     return trapmf([x], 29, 31, 100, 100)[0]

# Uniwersum i zbiory rozmyte dla wyjścia (Ryzyko 0-100)
RISK = np.linspace(0, 100, 101)
risk_low    = trimf(RISK, 0, 20, 40)
risk_medium = trimf(RISK, 30, 50, 70)
risk_high   = trimf(RISK, 60, 80, 100)

def explainable_fuzzy_obesity(patient):
    bmi = patient["bmi"]
    

    mu = {
        "BMI_normalne": bmi_normalne(bmi),
        "BMI_podwyzszone": bmi_podwyzszone(bmi),
        "BMI_wysokie": bmi_wysokie(bmi)
    }
    
    rules =[
        {
            "id": "R1",
            "text": "Jeśli BMI jest WYSOKIE, to ryzyko otyłości jest WYSOKIE.",
            "strength": mu["BMI_wysokie"],
            "out_set": risk_high,
            "why": f"Przynależność BMI ({bmi}) do zbioru 'wysokie' wynosi {mu['BMI_wysokie']:.2f}"
        },
        {
            "id": "R2",
            "text": "Jeśli BMI jest PODWYŻSZONE, to ryzyko otyłości jest ŚREDNIE.",
            "strength": mu["BMI_podwyzszone"],
            "out_set": risk_medium,
            "why": f"Przynależność BMI ({bmi}) do zbioru 'podwyższone' wynosi {mu['BMI_podwyzszone']:.2f}"
        },
        {
            "id": "R3",
            "text": "Jeśli BMI jest NORMALNE, to ryzyko otyłości jest NISKIE.",
            "strength": mu["BMI_normalne"],
            "out_set": risk_low,
            "why": f"Przynależność BMI ({bmi}) do zbioru 'normalne' wynosi {mu['BMI_normalne']:.2f}"
        }
    ]
    

    clipped =[]
    for r in rules:
        clipped_set = np.minimum(r["strength"], r["out_set"])
        clipped.append(clipped_set)
        r["clipped_area"] = float(clipped_set.sum())
        
    aggregated = np.maximum.reduce(clipped) if clipped else np.zeros_like(RISK)
    
    if aggregated.sum() == 0:
        crisp = 0.0
    else:
        crisp = float((RISK * aggregated).sum() / aggregated.sum())
        
    total_area = sum(r["clipped_area"] for r in rules) + 1e-12
    for r in rules:
        r["contribution_pct"] = 100.0 * r["clipped_area"] / total_area
        
    rules_sorted = sorted(rules, key=lambda r: r["strength"], reverse=True)
    label = "niskie" if crisp < 35 else ("średnie" if crisp < 65 else "wysokie")
    
    return {
        "patient": patient,
        "memberships": mu,
        "rules_fired": rules_sorted,
        "risk_crisp": crisp,
        "risk_label": label
    }

def print_report(exp):
    p = exp["patient"]
    print(f"PACJENT {p['patient_id']} | BMI: {p['bmi']}")
    print(f"Decyzja klasyczna : {classical_obesity_diagnosis(p)}")
    print(f"Decyzja rozmyta   : Ryzyko {exp['risk_crisp']:.1f}/100 ({exp['risk_label'].upper()})")
    
    print("Wyjaśnienie (XAI):")
    for r in exp["rules_fired"]:
        if r["strength"] > 0:
            print(f" -> [{r['id']}] Aktywacja: {r['strength']:.2f} (Wpływ: ~{r['contribution_pct']:.0f}%)")
            print(f"    Dlaczego: {r['why']}")
    print("-" * 60)


def process_all_patients(csv_filename):

    print("URUCHAMIANIE SYSTEMU EKSPERTOWEGO - WARIANT 4 (OTYŁOŚĆ)")
    
    # Otwieramy i czytamy plik z danymi
    with open(csv_filename, mode='r', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        
        for row in reader:

            patient = {
                "patient_id": row["patient_id"],
                "age": int(row["age"]),
                "bmi": float(row["bmi"]),
                "glucose": int(row["glucose"]),
                "systolic_bp": int(row["systolic_bp"]),
                "diastolic_bp": int(row["diastolic_bp"])
            }
            
            explanation = explainable_fuzzy_obesity(patient)
            print_report(explanation)


if __name__ == "__main__":
    process_all_patients("pacjenci_demo_system_ekspertowy.csv")

URUCHAMIANIE SYSTEMU EKSPERTOWEGO - WARIANT 4 (OTYŁOŚĆ)
PACJENT P01 | BMI: 21.7
Decyzja klasyczna : Not Obese (Brak otyłości)
Decyzja rozmyta   : Ryzyko 20.0/100 (NISKIE)
Wyjaśnienie (XAI):
 -> [R3] Aktywacja: 1.00 (Wpływ: ~100%)
    Dlaczego: Przynależność BMI (21.7) do zbioru 'normalne' wynosi 1.00
------------------------------------------------------------
PACJENT P02 | BMI: 27.8
Decyzja klasyczna : Not Obese (Brak otyłości)
Decyzja rozmyta   : Ryzyko 50.0/100 (ŚREDNIE)
Wyjaśnienie (XAI):
 -> [R2] Aktywacja: 1.00 (Wpływ: ~100%)
    Dlaczego: Przynależność BMI (27.8) do zbioru 'podwyższone' wynosi 1.00
------------------------------------------------------------
PACJENT P03 | BMI: 21.2
Decyzja klasyczna : Not Obese (Brak otyłości)
Decyzja rozmyta   : Ryzyko 20.0/100 (NISKIE)
Wyjaśnienie (XAI):
 -> [R3] Aktywacja: 1.00 (Wpływ: ~100%)
    Dlaczego: Przynależność BMI (21.2) do zbioru 'normalne' wynosi 1.00
------------------------------------------------------------
PACJENT P04 | BMI: 